In [ ]:
import zipfile
import io
import os
import shutil
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Paths
drive_zip_path = '/content/drive/MyDrive/BT4222_Project/html_MyAnimelist.zip'
local_zip_path = '/content/html_MyAnimelist.zip'
output_csv = '/content/drive/MyDrive/BT4222_Project/anime_metadata_final.csv'

# 3. Copy zip to Colab if not already there
if not os.path.exists(local_zip_path):
    print("Copying zip file to local runtime...")
    shutil.copy2(drive_zip_path, local_zip_path)

data_list = []

print("Starting combined HTML parsing...")
with zipfile.ZipFile(local_zip_path, 'r') as outer_zip:
    # Filter out __MACOSX files from the list of inner zips
    inner_zips = [name for name in outer_zip.namelist() if name.endswith('.zip') and not name.startswith('__MACOSX/')]

    # --- ⚠️ REMOVE '[:100]' BELOW ONCE YOU ARE READY TO RUN ALL 48,492 FILES ---
    # Try it with [:100] first to verify the Characters and Staff are extracting properly.
    for inner_zip_name in tqdm(inner_zips):
        anime_id = inner_zip_name.split('.')[0].split('/')[-1]
        inner_bytes = outer_zip.read(inner_zip_name)

        with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as inner_zip:
            file_names = inner_zip.namelist()

            # Initialize empty strings so empty data doesn't break the script
            synopsis, background, genres_str, studios_str = "", "", "", ""
            characters_str, staff_str = "", ""

            # --- FIND DETAILS.HTML ---
            details_file = next((f for f in file_names if 'details.html' in f), None)
            if details_file:
                html_content = inner_zip.read(details_file).decode('utf-8', errors='ignore')
                soup = BeautifulSoup(html_content, 'lxml')

                # Synopsis & Background
                syn_tag = soup.find('p', itemprop='description')
                if syn_tag: synopsis = syn_tag.get_text(separator=' ', strip=True)

                bg_header = soup.find('h2', string='Background')
                if bg_header and bg_header.parent:
                    bg_parts = []
                    for node in bg_header.parent.next_siblings:
                        if getattr(node, 'name', None) == 'div': break
                        if isinstance(node, str): bg_parts.append(node.strip())
                    background = ' '.join(filter(None, bg_parts))

                # Genres & Studios
                genres = [span.get_text(strip=True) for span in soup.find_all('span', itemprop='genre')]
                genres_str = ', '.join(genres)

                studio_span = soup.find('span', class_='dark_text', string='Studios:')
                if studio_span and studio_span.parent:
                    studios = [a.get_text(strip=True) for a in studio_span.parent.find_all('a')]
                    studios_str = ', '.join(studios)

            # --- FIND STAFF.HTML ---
            staff_file = next((f for f in file_names if 'staff.html' in f), None)
            if staff_file:
                html_content = inner_zip.read(staff_file).decode('utf-8', errors='ignore')
                soup = BeautifulSoup(html_content, 'lxml')

                # Characters (Grab names from links pointing to /character/)
                char_links = soup.find_all('a', href=lambda href: href and '/character/' in href)
                characters = list(set([a.get_text(strip=True) for a in char_links if a.get_text(strip=True)]))
                characters_str = ', '.join(characters[:15])

                # Staff/Voice Actors (Grab names from links pointing to /people/)
                staff_links = soup.find_all('a', href=lambda href: href and '/people/' in href)
                staff_members = list(set([a.get_text(strip=True) for a in staff_links if a.get_text(strip=True)]))
                staff_str = ', '.join(staff_members[:15])

            # Save combined row
            data_list.append({
                'MAL_ID': anime_id,
                'Synopsis': synopsis,
                'Background': background,
                'Genres': genres_str,
                'Studios': studios_str,
                'Characters': characters_str,
                'Staff': staff_str
            })

df = pd.DataFrame(data_list)
df.to_csv(output_csv, index=False)
print(f"\nDone! Final combined metadata saved to: {output_csv}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting combined HTML parsing...


  0%|          | 0/17562 [00:00<?, ?it/s]


Done! Final combined metadata saved to: /content/drive/MyDrive/BT4222_Project/anime_metadata_final.csv


In [ ]:
import pandas as pd

# 1. Load your newly generated dataset
df_scraped = pd.read_csv('/content/drive/MyDrive/BT4222_Project/anime_metadata_final.csv')

# 2. Drop the columns that came back empty
df_scraped = df_scraped.drop(columns=['Background', 'Studios'])

# 3. Fill the few missing synopses and staff with an empty string
df_scraped['Synopsis'] = df_scraped['Synopsis'].fillna('')
df_scraped['Staff'] = df_scraped['Staff'].fillna('')

# 4. Save the final, clean version!
clean_path = '/content/drive/MyDrive/BT4222_Project/anime_metadata_ready.csv'
df_scraped.to_csv(clean_path, index=False)
print("Cleaned data shape:", df_scraped.shape)

Cleaned data shape: (17562, 5)
